# Заняття 6 · Демо 2 · Open-vocabulary детектори: клас — це рядок тексту

**LIVE DEMO 2 (слайд 43)** · слайди 34–42, 45–46, 49–51

Вузол «знайти й назвати» в Grounded-SAM. Замість фіксованої голови на N класів — зіставлення регіонів кадру з текстом
у спільному просторі: $S = O P^\top$ (слайд 34). Розбираємо чотири гілки підходів на одних і тих самих кадрах
VisDrone-val і міряємо, що з цього доходить до аерознімки:

| | Гілка | Моделі | Що важливо інженерно |
|---|---|---|---|
| 02 | aligned: CLIP-бекбон + легкі голови | OWLv2, YOLO-World, YOLOE | текст зустрічається з кадром лише в кінці → новий запит майже безкоштовний |
| 03 | grounding: злиття модальностей | Grounding DINO T / B | текст впливає на ознаки → кожна зміна тексту — повний прохід |
| 04 | VLM-grounding: бокси як текст | Florence-2, Qwen2.5-VL (опційно) | найгнучкіший запит, секунди на кадр, жодного score |

| Розділ | Що показуємо | Слайди |
|---|---|---|
| 1. S = O·Pᵀ руками | Ембеддинги патчів і запитів OWLv2, теплові карти, пастка з квадратом | 34, 36 |
| 2. Скільки коштує новий запит | Aligned проти grounding проти «запеченого» словника | 35, 37, 38, 41 |
| 3. Механіка текстового промпта | Крапка проти коми, два пороги, ліміт 256 токенів | 39 |
| 4. Пікселі до моделі | GSD → пікселі на цілі для кожної моделі; формула тайлів | 45, 46 |
| 5. Чесний бенчмарк | AP / AP_small / мс: усі кадри і кадри з дрібними цілями, повний кадр проти тайлів | 35, 45, 46, 49, 50 |
| 6. Промпт — це конфігурація | Формулювання класу, ансамбль, негативні класи, dev / test | 40, 50 |
| 7. Об'єкт без назви | Запит зразком: ембеддинг зразка в OWLv2, YOLOE visual prompt | 36, 41, 51, 62 |
| 8. VLM пише бокси текстом | Florence-2 / Qwen2.5-VL: пастка координат, час, натовп | 42 |

**Підготовка (один раз, з мережею):**
```bash
# з кореня репозиторію
python tools/fetch_visdrone.py --dataset visdrone-det   # VisDrone2019-DET-val, ~80 МБ
python tools/fetch_ov_models.py                         # GDINO-T/B, OWLv2, Florence-2, YOLOE, YOLO-World, YOLO11
python tools/fetch_ov_models.py --models qwen2.5-vl     # опційно, ≈ 7.5 ГБ — для розділу 8
```
Повний бенчмарк (розділ 5) — ~10 хвилин на GPU; результат кешується в `outputs/demo2/02_ov_bench.csv`, і повторний запуск
(`RECOMPUTE = False`) бере його. На лекції або показуйте збережений, або ставте `N_IMG = 30`.

In [ ]:
from pathlib import Path

# ---------------- Конфігурація ----------------
DEVICE = None
N_IMG = 100                    # кадрів VisDrone-DET-val для бенчмарку (розділ 5); 30 — для живого показу
N_SMALL = 20                   # кадрів ≥ 1900 px з найбільшою кількістю цілей < 20 px — набір, де тайли мають сенс
RECOMPUTE = False              # False — взяти збережений 02_ov_bench.csv, якщо він на тих самих наборах
BENCH = ["gdino-tiny", "gdino-base", "owlv2", "yoloworld", "yoloe"]
CLASSES = ["person", "car", "truck", "bus"]          # VisDrone: person = pedestrian + people, car = car + van
TILE, OVERLAP = 640, 0.2
SEED = 0
OUT = Path("outputs/demo2")

import panels as P
VLM = ["florence2"] + (["qwen2.5-vl"] if P.have("qwen2.5-vl") else [])
hf = P.setup_hf(ov=BENCH + VLM)

import time
import numpy as np, pandas as pd, torch, cv2
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
pd.set_option("display.precision", 3)
DEVICE = P.pick_device(DEVICE)
GPU = P.gpu_name(DEVICE)
OUT.mkdir(parents=True, exist_ok=True)

items = P.det_items(n=N_IMG, seed=SEED)
# демо-кадр: машини, автобуси й люди одночасно, середній розмір
def mix(it):
    p, gt = it
    return min((gt.cats == 9).sum(), 4) * 3 + min(np.isin(gt.cats, [4, 5]).sum(), 30) + min(np.isin(gt.cats, [1, 2]).sum(), 10)
DEMO_PATH, GTD = max(P.det_items(), key=mix)
img = P.imread_rgb(DEMO_PATH)
H, W = img.shape[:2]
print(f"HF: {hf} | {DEVICE}: {GPU} | бенчмарк: {len(items)} кадрів | демо-кадр {DEMO_PATH.name} {W}×{H} | VLM: {VLM}")

## 1. S = O·Pᵀ руками (OWLv2)

OWLv2 — CLIP-ViT без пулінгу: **кожен патч-токен — кандидат на об'єкт** зі своїм боксом і своїм ембеддингом (слайд 36).
- $O \in \mathbb{R}^{N \times d}$ — L2-нормовані ембеддинги регіонів (N = 60 × 60 патчів для входу 960²);
- $P \in \mathbb{R}^{M \times d}$ — L2-нормовані ембеддинги текстових запитів;
- $S = O P^\top$ — косинус кожного регіону з кожним запитом. Той самий скалярний добуток, що в InfoNCE CLIP із заняття 5,
  але по регіонах, а не по кадру. Далі в кожного патча свій зсув і масштаб: $\text{logit} = (S + b_i)\,s_i$, $p = \sigma(\text{logit})$.

In [ ]:
owl = P.Owl("owlv2", DEVICE)
QUERIES = ["a photo of a car", "a photo of a bus", "a photo of a person", "a photo of a tree"]

O, (gh, gw) = owl.patch_embeds(img)          # регіони кадру
Pm = owl.query_matrix(QUERIES)               # токени запиту
S = O @ Pm.T                                 # ← S = O·Pᵀ
logits = owl.calibrate(S, img)               # (S + shift) · scale — окремо для кожного патча
print(f"O {O.shape} · Pᵀ {Pm.T.shape} = S {S.shape}; cos ∈ [{S.min():.2f}, {S.max():.2f}]")

# перевірка: те саме, що віддає повний forward моделі
import PIL
inp = owl.proc(text=[QUERIES], images=PIL.Image.fromarray(img), return_tensors="pt").to(DEVICE)
with torch.inference_mode():
    ref = owl.model(**inp).logits[0].float().cpu().numpy()
print(f"max |logits руками − logits моделі| = {np.abs(ref - logits).max():.1e}")

# теплові карти: лише частина сітки, що лежить на кадрі (решта — паддінг до квадрата)
rows, cols = int(np.ceil(gh * H / max(H, W))), int(np.ceil(gw * W / max(H, W)))
maps = []
for j, q in enumerate(QUERIES):
    prob = 1 / (1 + np.exp(-logits[:, j].reshape(gh, gw)[:rows, :cols]))
    hm = cv2.resize((plt.cm.inferno(prob / max(prob.max(), 1e-6))[..., :3] * 255).astype(np.uint8), (W, H),
                    interpolation=cv2.INTER_NEAREST)
    maps.append((0.45 * img + 0.55 * hm).astype(np.uint8))
P.show_row(maps, [f"«{q}» · max p = {1 / (1 + np.exp(-logits[:, j].max())):.2f}" for j, q in enumerate(QUERIES)],
           size=4.2, ncols=4, suptitle="σ((S + shift)·scale) по патч-токенах: кожна клітинка — кандидат на об'єкт",
           save=OUT / "02_owl_patch_maps.png")

**Пастка (слайд 36).** OWLv2 доповнює кадр до квадрата справа / знизу, тож координати боксів — у квадраті S × S,
S = max(H, W). Якщо передати `target_size = (H, W)`, усі бокси по вертикалі стиснуться — і це легко прийняти за «погану модель».

In [ ]:
q = ["a photo of a car", "a photo of a bus"]
good, bad = owl(img, q, thr=0.25), owl(img, q, thr=0.25, square=False)
vis = P.draw_boxes(img, bad.boxes, color=(240, 70, 70))
vis = P.draw_boxes(vis, good.boxes, color=(40, 220, 170))
P.show_row([vis], ["бірюзові — target_size = (S, S), правильно; червоні — (H, W): бокси з'їхали вгору й стиснулись"], size=12)

**Підсумок розділу.** Open-vocabulary детектор — це класифікатор регіонів, у якого замість вагової матриці останнього
шару стоїть матриця текстових ембеддингів $P$. Додати клас = дописати рядок у $P$ (слайд 34). Ціна — якість тепер залежить
від формулювання (розділ 6), а «запит про дерево» дасть дерева: модель відповідає на те, що спитали, а не на те, що треба.

## 2. Скільки коштує новий запит

Гілки відрізняються тим, **де** текст зустрічається з кадром (слайди 35, 37, 41):
* **aligned (OWLv2):** ознаки кадру не залежать від тексту → рахуємо раз, новий словник — лише текстовий енкодер + скалярні добутки;
* **grounding (Grounding DINO):** текст змінює ознаки кадру в кількох шарах → новий словник = повний прохід;
* **запечений словник (YOLOE / YOLO-World):** текст → ембеддинги один раз «на землі», далі звичайний YOLO без текстового енкодера.

In [ ]:
gd = P.GDino("gdino-base", DEVICE)
yoloe = P.load_ultra("yoloe", CLASSES)
VOCABS = [["car"], ["truck", "bus"], ["person", "bicycle"], ["van", "tricycle", "motorcycle"], ["tree", "building"]]


def ms(fn, n=5):
    fn()
    return P.bench(fn, n_warmup=1, n_iter=n)["median_ms"]


img2 = img.copy()
rows = []
rows.append({"модель": "OWLv2 (aligned)",
             "новий кадр, мс": ms(lambda: owl.image_features(img2, cache=False)),
             "новий словник на тому ж кадрі, мс": np.median([ms(lambda v=v: owl(img, [f"a photo of a {c}" for c in v]))
                                                             for v in VOCABS]),
             "де працює текстовий енкодер": "на кожну зміну словника, дешево"})
rows.append({"модель": "Grounding DINO-B (grounding)",
             "новий кадр, мс": ms(lambda: gd(img, " . ".join(CLASSES) + " .")),
             "новий словник на тому ж кадрі, мс": np.median([ms(lambda v=v: gd(img, " . ".join(v) + " .")) for v in VOCABS]),
             "де працює текстовий енкодер": "у кожному проході: текст змінює ознаки"})
t_bake = []
for v in VOCABS:
    with P.timed(t_bake):
        P.set_ultra_classes(yoloe, v)
P.set_ultra_classes(yoloe, CLASSES)
rows.append({"модель": "YOLOE-11L (запечений словник)",
             "новий кадр, мс": ms(lambda: P.ultra_detect(yoloe, img)),
             "новий словник на тому ж кадрі, мс": float(np.median(t_bake)) + ms(lambda: P.ultra_detect(yoloe, img)),
             "де працює текстовий енкодер": "лише при set_classes, на землі; на борту його немає"})
cost = pd.DataFrame(rows).set_index("модель")
cost.to_csv(OUT / "02_vocab_change_cost.csv")
cost.round(1)

**Підсумок розділу.** Перебирати класи «на льоту» дешево лише в aligned-гілці; у Grounding DINO кожен варіант промпта —
повний прохід мережею (слайди 37–38). YOLOE / YOLO-World найшвидші на кадр, але змінити словник у полі без
текстового енкодера неможливо (слайд 41): репараметризація — це той самий «закритий словник», лише зібраний за хвилину.

## 3. Механіка текстового промпта (Grounding DINO)

Три деталі зі слайду 39:
* **Формат:** `car . truck . bus .` — крапка відділяє категорії маскою уваги BERT. З комою це одна фраза.
* **Два пороги:** `box_threshold` (у transformers — `threshold`) вирішує, *чи є* об'єкт; `text_threshold` — *як його назвати*.
* **Ліміт 256 токенів BERT:** довгий словник мовчки обрізається.

In [ ]:
fmt = []
for prompt in ["car . truck . bus .", "car, truck, bus", "car truck bus"]:
    for bt in (0.35, 0.2):
        d = gd(img, prompt, classes=["car", "truck", "bus"], box_thr=bt)
        fmt.append({"промпт": prompt, "box_threshold": bt, "боксів": len(d), "макс. score": d.scores.max() if len(d) else 0,
                    "мітки моделі": dict(pd.Series(d.names, dtype=str).value_counts().head(3))})
display(pd.DataFrame(fmt).set_index(["промпт", "box_threshold"]))

grid = []
for bt in [0.15, 0.25, 0.35, 0.5]:
    for tt in [0.15, 0.25, 0.5]:
        d = gd(img, "car . truck . bus .", box_thr=bt, text_thr=tt)
        grid.append({"box_threshold": bt, "text_threshold": tt, "боксів": len(d),
                     "без мітки або склеєна": int(sum(n.strip() not in ("car", "truck", "bus") for n in d.names))})
grid = pd.DataFrame(grid)
display(grid.pivot(index="box_threshold", columns="text_threshold", values=["боксів", "без мітки або склеєна"]))

d15, d35 = gd(img, "car . truck . bus .", box_thr=0.15), gd(img, "car . truck . bus .", box_thr=0.35)
P.show_row([P.draw_boxes(img, d15.boxes), P.draw_boxes(img, d35.boxes)],
           [f"box_threshold 0.15: {len(d15)} боксів — сміття, яке піде в SAM", f"box_threshold 0.35: {len(d35)} боксів"],
           size=6.5, save=OUT / "02_threshold_garbage.png")

# ліміт 256 токенів: словник COCO-80 + 40 багатослівних описів під ISR
COCO80 = ("person bicycle car motorcycle airplane bus train truck boat traffic-light fire-hydrant stop-sign parking-meter "
          "bench bird cat dog horse sheep cow elephant bear zebra giraffe backpack umbrella handbag tie suitcase frisbee "
          "skis snowboard sports-ball kite baseball-bat baseball-glove skateboard surfboard tennis-racket bottle wine-glass "
          "cup fork knife spoon bowl banana apple sandwich orange broccoli carrot hot-dog pizza donut cake chair couch "
          "potted-plant bed dining-table toilet tv laptop mouse remote keyboard cell-phone microwave oven toaster sink "
          "refrigerator book clock vase scissors teddy-bear hair-drier toothbrush").replace("-", " ").split(" ")
ISR = [f"{a} {b}" for a in ["small", "large", "camouflaged", "tracked", "wheeled"]
       for b in ["military truck", "armored vehicle", "pickup truck with a mounted gun", "fuel tanker", "radar station",
                 "artillery piece", "trench line", "drone on the ground"]]
vocab = COCO80 + ISR
tok = gd.proc.tokenizer
for name, words in [("COCO-80", COCO80), (f"COCO-80 + {len(ISR)} ISR-описів", vocab)]:
    n_tok = len(tok(" . ".join(words) + " .")["input_ids"])
    fits = 0
    for i in range(1, len(words) + 1):                           # скільки класів повністю влазить у 256 токенів
        if len(tok(" . ".join(words[:i]) + " .")["input_ids"]) <= 256:
            fits = i
    print(f"{name}: {len(words)} класів = {n_tok} токенів BERT → у 256 влазить {fits}; "
          f"{'решту модель мовчки не побачить, наприклад: ' + ', '.join(words[fits:fits + 3]) if fits < len(words) else 'усе влазить'}")

**Підсумок розділу.** З комою модель бачить одну фразу, і мітки «склеюються» або зникають. `box_threshold` змінює кількість
боксів, `text_threshold` — лише мітки. Занижений `box_threshold` — найдешевший спосіб отруїти датасет: SAM слухняно
обведе кожен сміттєвий бокс (демо 3). Великий словник ріжте на кілька проходів — і платіть за кожен (розділ 2).

## 4. Скільки пікселів доходить до моделі

Розрахунок, який треба зробити **до** вибору моделі (слайд 45). Умовний борт: висота 300 м, фокус 12 мм, піксель 2.9 мкм,
кадр 4K. Ground sampling distance $\text{GSD} = h \cdot p / f$, ціль $L$ метрів займає $L / \text{GSD}$ пікселів на сирому
кадрі, а далі кожна модель стискає кадр до свого входу.

In [ ]:
H_M, F_MM, PIX_UM = 300, 12, 2.9                  # висота, фокус, піксель сенсора
FRAME = (3840, 2160)
L_M = 4.5                                         # довжина легкової машини
GSD = H_M * PIX_UM * 1e-6 / (F_MM * 1e-3)         # м / px
p_raw = L_M / GSD
print(f"GSD = {GSD * 100:.2f} см/px → машина {L_M} м = {p_raw:.1f} px на сирому 4K-кадрі")

rows = [{"модель / вхід": "сирий кадр 4K", "масштаб": 1.0, "px на цілі": p_raw}]
for name, (kind, res) in P.INPUT_RES.items():
    if kind == "gdino":
        scale = min(res[0] / min(FRAME), res[1] / max(FRAME))     # коротша → 800, але довша ≤ 1333
    else:
        scale = res / max(FRAME)
    rows.append({"модель / вхід": name, "масштаб": scale, "px на цілі": p_raw * scale})
rows.append({"модель / вхід": f"тайл {TILE} (без ресайзу)", "масштаб": 1.0, "px на цілі": p_raw})
px = pd.DataFrame(rows).set_index("модель / вхід")
display(px.round(2))

fig, ax = plt.subplots(figsize=(8, 3.4))
colors = ["#2bb5a0" if v >= 60 else "#3aa0e8" if v >= 20 else "#e0703a" for v in px["px на цілі"]]
ax.barh(px.index[::-1], px["px на цілі"][::-1], color=colors[::-1])
for y, v in enumerate(px["px на цілі"][::-1]):
    ax.text(v + 0.5, y, f"{v:.1f}", va="center", fontsize=8)
ax.axvline(20, color="#777", ls=":", lw=1); ax.text(20.5, len(px) - 0.6, "≥ 20 px — детекція", fontsize=8, color="#555")
ax.axvline(60, color="#777", ls=":", lw=1); ax.text(60.5, len(px) - 0.6, "60–80 px — тип", fontsize=8, color="#555")
ax.set_xlabel(f"пікселів по довжині машини {L_M} м після ресайзу")
P.save_fig(fig, OUT / "02_pixels_per_model.png")

# Формула тайлів (слайд 46): n = ceil((W − t) / (t (1 − o))) + 1
for (Wf, Hf) in [(3840, 2160), (1920, 1080), (1360, 765)]:
    nx = int(np.ceil((Wf - TILE) / (TILE * (1 - OVERLAP)))) + 1
    ny = int(np.ceil((Hf - TILE) / (TILE * (1 - OVERLAP)))) + 1
    print(f"{Wf}×{Hf}, t = {TILE}, o = {OVERLAP}: {nx} × {ny} = {nx * ny} тайлів (+1 повний кадр у SAHI); "
          f"у модулі: {len(P.tile_origins(Wf, Hf, TILE, OVERLAP))}")

**Підсумок розділу.** Та сама машина — 62 px на сирому 4K, але 15–22 px на вході більшості OV-моделей і 4 px у CLIP 224².
Тайл 640 зберігає 100 % роздільності ціною ×32 проходів на 4K-кадр. Тому спершу $p'$, потім модель (слайд 65, «пікселі до моделі»).

## 5. Чесний бенчмарк на VisDrone-val

Однаковий протокол для всіх: ті самі кадри, ті самі 4 класи, ignored regions VisDrone не штрафуються, пороги навмисно
низькі (для AP потрібна вся крива precision / recall). Два набори: **випадкові `N_IMG` кадрів VisDrone-DET-val**
(здебільшого 1360 × 765, цілі різного розміру) і **кадри ≥ 1900 px з найбільшою кількістю цілей < 20 px** — те, заради чого
тайли існують. Два режими: **повний кадр** і **тайли 640 / 20 % +
повний кадр** зі склейкою NMS і злиттям вкладених боксів (SAHI). Метрики: AP, AP50, **AP_small** (< 32² px — саме те, що
важить на аерознімці, слайд 50) і мс на кадр.

In [ ]:
OWL_QUERIES = [f"a photo of a {c}" for c in CLASSES]
GD_PROMPT = " . ".join(CLASSES) + " ."


def make_detector(name):
    # повертає (fn_full, fn_tile, close): fn(img) -> P.Dets з labels = індекс у CLASSES
    if name.startswith("gdino"):
        g = gd if name == "gdino-base" else P.GDino(name, DEVICE)
        f = lambda im: g(im, GD_PROMPT, CLASSES, box_thr=0.1, text_thr=0.1)
        return f, f, g
    if name == "owlv2":
        f = lambda im: owl(im, OWL_QUERIES, thr=0.05)
        return f, f, owl
    m = yoloe if name == "yoloe" else P.load_ultra(name, CLASSES)
    return (lambda im: P.ultra_detect(m, im, conf=0.01, imgsz=1280),       # повний кадр: вхід 1280
            lambda im: P.ultra_detect(m, im, conf=0.01, imgsz=TILE), m)     # тайл 640 — без апсемплу до 1280


def run(name, fn_full, fn_tile, mode, items):
    preds, ts, passes = {}, [], []
    fn_full(P.imread_rgb(items[0][0]))                                      # прогрів
    for p, _ in tqdm(items, desc=f"{name} · {mode}", leave=False):
        im = P.imread_rgb(p)
        with P.timed(ts):
            if mode == "повний кадр":
                d, n = fn_full(im), 1
            else:
                d, n = P.detect_tiled(fn_tile, im, TILE, OVERLAP, with_full=True)
        preds[p.stem] = d
        passes.append(n)
    m = P.coco_eval(items, preds, CLASSES)
    return {"модель": P.OV_ZOO[name]["label"], "режим": mode, **m, "мс на кадр": float(np.median(ts)),
            "проходів": float(np.mean(passes))}


def n_small(it):
    p, gt = it
    return (np.maximum(gt.boxes[:, 2] - gt.boxes[:, 0], gt.boxes[:, 3] - gt.boxes[:, 1]) < 20).sum()


big = [it for it in P.det_items() if P.image_size(it[0])[0] >= 1900]
SETS = {"усі кадри": items, "≥ 1900 px, дрібні цілі": sorted(big, key=lambda it: -n_small(it))[:N_SMALL]}
sig = "; ".join(f"{k}: {len(v)}" for k, v in SETS.items())                # підпис наборів для кешу
csv = OUT / "02_ov_bench.csv"
cached = pd.read_csv(csv) if csv.exists() else None
if not RECOMPUTE and cached is not None and "набори" in cached and cached["набори"].iloc[0] == sig:
    res = cached
    print(f"взято збережений {csv} ({res['залізо'].iloc[0]})")
else:
    rows = []
    for name in BENCH:
        ff, ft, model = make_detector(name)
        for set_name, its in SETS.items():
            for mode in ["повний кадр", "тайли"]:
                rows.append({"набір": set_name, **run(name, ff, ft, mode, its)})
                print({k: (round(v, 3) if isinstance(v, float) else v) for k, v in rows[-1].items()})
        if model not in (gd, owl, yoloe):
            del model
            P.free_memory()
    res = pd.DataFrame(rows)
    res["набори"], res["залізо"] = sig, GPU
    res.to_csv(csv, index=False)
display(res.pivot_table(index="модель", columns=["набір", "режим"], values=["AP_small", "мс на кадр"], sort=False).round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.4), sharey=False)
for ax, (set_name, rs) in zip(axes, res.groupby("набір", sort=False)):
    for i, (name, g) in enumerate(rs.groupby("модель", sort=False)):
        f, t = g[g["режим"] == "повний кадр"].iloc[0], g[g["режим"] == "тайли"].iloc[0]
        c = plt.cm.tab10(i)
        ax.annotate("", xy=(t["мс на кадр"], t.AP50), xytext=(f["мс на кадр"], f.AP50),
                    arrowprops=dict(arrowstyle="->", color=c, lw=1.2))
        ax.scatter(f["мс на кадр"], f.AP50, color=c, s=50, zorder=3, label=name)
        ax.scatter(t["мс на кадр"], t.AP50, color=c, s=50, marker="s", zorder=3)
    ax.set_xscale("log"); ax.set_xlabel("мс на кадр (медіана, log)"); ax.set_ylabel("AP50")
    ax.set_title(f"{set_name}: ● повний кадр → ■ тайли {TILE} / {int(OVERLAP * 100)} %", fontsize=10)
axes[0].legend(fontsize=8, loc="lower right")
fig.suptitle(f"Точність / час, zero-shot · {res['залізо'].iloc[0]}")
P.save_fig(fig, OUT / "02_accuracy_vs_latency.png")

piv = res.pivot_table(index="модель", columns=["набір", "режим"], values="AP_small", sort=False)
for set_name in SETS:
    piv[(set_name, "приріст від тайлів")] = piv[(set_name, "тайли")] - piv[(set_name, "повний кадр")]
display(piv[[c for s_ in SETS for c in piv.columns if c[0] == s_]].round(3))

ours = res[(res["режим"] == "повний кадр") & (res["набір"] == "усі кадри")].set_index("модель")["AP"] * 100
print("COCO zero-shot у статті → VisDrone-val тут (той самий чекпойнт, AP):")
print(f"  Grounding DINO-T: 48.4 → {ours.get('Grounding DINO-T', float('nan')):.1f}")
print(f"  Grounding DINO-B: COCO був у його тренуванні, тож «zero-shot на COCO» для нього не існує → {ours.get('Grounding DINO-B', float('nan')):.1f}")

**Підсумок розділу.**
* Дві цифри про одну модель — обидві правдиві (слайд 49): ~48 AP на COCO і ~20 на VisDrone. Публічний AP — не ваш AP.
* Тайли — не безкоштовна магія. На типових кадрах ~1360 px тайл 640 майже не збільшує ціль (вхід моделі й так ~800–1280),
  зате додає обрізані бокси на межах і хибні спрацювання на фоні без контексту сцени — AP падає. На кадрах, де цілі дрібні
  **в пікселях**, тайли повертають AP_small — але не всім моделям однаково, і платите ви кратністю проходів. Великий кадр
  сам по собі нічого не означає: 4K з малої висоти — це великі цілі. Спершу рахуйте $p'$ (розділ 4), потім вирішуйте, чи
  потрібні тайли (слайди 45–46), і перевіряйте на своїх кадрах.
* Читайте чекпойнт, а не назву: `grounding-dino-base` бачив COCO в тренуванні, тож для COCO він не zero-shot (слайд 7,
  «термінологічна гігієна»). Для вашого домену — так само: питайте, що було в претренуванні.

## 6. Промпт — це конфігурація системи

Один клас — `truck` (VisDrone: truck), одна модель — Grounding DINO-B, кілька формулювань (слайд 40). Протокол чесний
(слайд 50): **кращий варіант обираємо на dev-кадрах, звітуємо на інших test-кадрах**. Друга частина — негативні класи:
до промпта `car` дописуємо те, з чим його плутають, і викидаємо бокси з цими мітками.

In [ ]:
dev = P.det_items(n=60, seed=SEED + 1)
test = P.det_items(n=60, seed=SEED + 1, offset=60)            # інші кадри тієї самої перестановки — не перетинаються
VARIANTS = {"truck": ["truck"], "lorry": ["lorry"], "vehicle": ["vehicle"],
            "large truck with cargo bed": ["large truck with cargo bed"],
            "truck seen from above": ["truck seen from above"],
            "ансамбль: truck . lorry . box truck . dump truck": ["truck", "lorry", "box truck", "dump truck"]}


def ap_truck(item_set, phrases):
    preds = {}
    for p, _ in item_set:
        d = gd(P.imread_rgb(p), " . ".join(phrases) + " .", phrases, box_thr=0.1, text_thr=0.1)
        d.labels = np.where(d.labels >= 0, 0, -1)                    # будь-який синонім = truck
        preds[p.stem] = P.nms(d, 0.5)
    return P.coco_eval(item_set, preds, ["truck"], groups={"truck": [6]})["AP"]


pv = pd.DataFrame([{"промпт": k, "AP dev": ap_truck(dev, v), "AP test": ap_truck(test, v)}
                   for k, v in tqdm(VARIANTS.items(), desc="варіанти")]).set_index("промпт")
best_dev, best_test = pv["AP dev"].idxmax(), pv["AP test"].idxmax()
pv.to_csv(OUT / "02_prompt_variants.csv")
pv["test − dev"] = pv["AP test"] - pv["AP dev"]
display(pv.style.format("{:+.3f}", subset=["test − dev"]).format("{:.3f}", subset=["AP dev", "AP test"])
        .highlight_max(subset=["AP dev", "AP test"], color="#c8efe6"))
print(f"обрали на dev: «{best_dev}» → чесна оцінка на test: AP {pv.loc[best_dev, 'AP test']:.3f}")
if best_dev != best_test:
    print(f"якби обирали на test: «{best_test}» → {pv.loc[best_test, 'AP test']:.3f} — оптимістична оцінка-витік")
print(f"розкид AP між dev і test для одного промпта: до {pv['test − dev'].abs().max():.3f} — це шум вибірки, "
      f"а не різниця між промптами")

In [ ]:
# Негативні класи: що «car» ловить на фоні і чи допомагає назвати це явно
NEG = ["building", "tree", "road marking", "shadow", "awning"]
rows = []
for name, phrases in [("car .", ["car"]), ("car . + негативні класи", ["car"] + NEG)]:
    pr = []
    for p, gt in test:
        d = gd(P.imread_rgb(p), " . ".join(phrases) + " .", phrases, box_thr=0.3, text_thr=0.25)
        keep = d.labels == 0                                        # лишаємо тільки бокси з міткою car
        pr.append(P.match_pr(d.boxes[keep], gt.boxes[np.isin(gt.cats, [4, 5])], gt.ignore))
    prec, rec = P.pr_table(pr)
    rows.append({"промпт": name, "precision": prec, "recall": rec, "хибних боксів": sum(r["fp"] for r in pr)})
pd.DataFrame(rows).set_index("промпт")

**Підсумок розділу.**
* Різниця між формулюваннями — того ж порядку, що між моделями. «vehicle» ловить і машини; фахова назва чи довгий опис
  часто гірші за просту візуальну фразу; ансамбль синонімів — дешевий спосіб не залежати від одного слова.
* Той самий промпт на двох підвибірках по 60 кадрів дає помітно різний AP — різниця між промптами одного порядку з цим шумом.
  Тому промпт і пороги обирають на dev і **заморожують до** заміру на test; підбір на тестових кадрах — той самий витік, що
  тренування на тесті (слайд 50). Версіонуйте промпт разом із кодом.
* Негативні класи змінюють і precision, і recall — у будь-який бік. Не вгадуйте: міряйте на своєму dev.

## 7. Об'єкт без назви: запит зразком

`awning-tricycle` — триколісний мотоцикл із тентом, клас VisDrone, якого немає в жодному вебовому словнику (аналог
«нового класу в полі», слайд 4). Порівнюємо текст і зразок: **один** чи **три** приклади з одного кадру → пошук на 12 інших
кадрах (слайди 36, 41). В OWLv2 зразок стає рядком матриці P замість тексту: ембеддинг патч-токенів, що лягають на бокс
зразка, — і далі той самий $S = O Q^\top$, що в розділі 1. Метрика — precision / recall за GT цього класу.

In [ ]:
pool = [(p, gt) for p, gt in P.det_items() if (gt.cats == 8).sum() >= 3]
pool.sort(key=lambda it: -(it[1].cats == 8).sum())
(ref_p, ref_gt), targets = pool[0], pool[1:13]
ref = P.imread_rgb(ref_p)
ex = ref_gt.boxes[ref_gt.cats == 8]
ex = ex[np.argsort(-(ex[:, 2] - ex[:, 0]) * (ex[:, 3] - ex[:, 1]))][:3]        # 3 найбільші приклади
crops = [ref[int(b[1]):int(b[3]), int(b[0]):int(b[2])] for b in ex]
P.show_row(crops, [f"зразок {i + 1}: {int(b[2] - b[0])}×{int(b[3] - b[1])} px" for i, b in enumerate(ex)], size=2.2)

yoloe_vp = P.load_ultra("yoloe")                                   # без тексту: класи задасть зразок
Q1, Q3 = owl.exemplar_embeds(ref, ex[:1]), owl.exemplar_embeds(ref, ex)   # рядки «матриці запитів» зі зразків
METHODS = {
    "GDINO-B, текст «awning tricycle»": lambda im: gd(im, "awning tricycle .", box_thr=0.3),
    "GDINO-B, текст «tricycle with a canopy»": lambda im: gd(im, "tricycle with a canopy .", box_thr=0.3),
    "OWLv2, 1 зразок (cos ≥ 0.75)": lambda im: owl.search(im, Q1, thr=0.75),
    "OWLv2, 3 зразки (cos ≥ 0.75)": lambda im: owl.search(im, Q3, thr=0.75),
    "YOLOE visual prompt, 1 зразок": lambda im: P.ultra_visual_prompt(yoloe_vp, im, ref, ex[:1], conf=0.25),
    "YOLOE visual prompt, 3 зразки": lambda im: P.ultra_visual_prompt(yoloe_vp, im, ref, ex, conf=0.25),
}
rows, shown = [], {}
for name, fn in METHODS.items():
    pr = []
    for p, gt in targets:
        im = P.imread_rgb(p)
        d = fn(im)
        pr.append(P.match_pr(d.boxes, gt.boxes[gt.cats == 8], gt.ignore))
        if p == targets[0][0]:
            shown[name] = P.draw_boxes(P.draw_boxes(im, gt.boxes[gt.cats == 8], color=(150, 150, 160)), d.boxes)
    prec, rec = P.pr_table(pr)
    rows.append({"метод": name, "precision": prec, "recall": rec, "знайдено": sum(r["tp"] + r["fp"] for r in pr)})
display(pd.DataFrame(rows).set_index("метод"))
P.show_row(list(shown.values()), list(shown), size=4.4, ncols=3,
           suptitle="сірі — GT awning-tricycle, бірюзові — знайдене", save=OUT / "02_query_by_example.png")

**Підсумок розділу.** Для класу без усталеної англійської назви текст майже мовчить, а зразок знаходить частину об'єктів
без жодного слова. Але **один зразок ≠ усі ракурси** (слайд 62): з одним прикладом recall мізерний, три помітно кращі —
і все одно precision низька, бо ціль у 30–50 px схожа на десятки інших дрібних об'єктів. Це не детектор, а генератор
кандидатів для людини: швидко набрати перші мітки, а далі — закритий детектор на них.

## 8. VLM пише бокси текстом

Четверта гілка (слайд 42): детекція як авторегресійна генерація тексту з координатами. Запит — довільна інструкція,
але **score немає**, час — секунди, а формат координат у кожної моделі свій:
* **Florence-2** повертає токени `<loc_0>…<loc_999>` — біни 0–999 по кожній осі кадру;
* **Qwen2.5-VL** — JSON з пікселями **зміненого** кадру (кратного 28 px), а не оригіналу.

Неврахований ресайз зсуває всі бокси — і це виглядає як «погана модель».

In [ ]:
all_items = P.det_items()
sparse_p, sparse_gt = next((p, gt) for p, gt in all_items if 2 <= (gt.cats == 9).sum() <= 6 and len(gt.boxes) < 40)
crowd_p, crowd_gt = next((p, gt) for p, gt in all_items                 # 1920 px і 30–50 машин: видно і натовп, і ресайз Qwen
                         if P.image_size(p)[0] >= 1900 and 30 <= np.isin(gt.cats, [4, 5]).sum() <= 50)
fl_proc, fl = P.load_hf("florence2", DEVICE, dtype=torch.float16 if DEVICE == "cuda" else None)

im = P.imread_rgb(sparse_p)
h, w = im.shape[:2]
t0 = time.perf_counter()
raw = P.florence_generate(fl_proc, fl, im, "<CAPTION_TO_PHRASE_GROUNDING>", "bus", device=DEVICE)
t_fl = time.perf_counter() - t0
names, bins = P.parse_loc_tokens(raw)
print(f"Florence-2, {t_fl:.2f} с. Сирий вихід:\n  {raw[:220]}…")
wrong = bins.copy()                                                    # біни прочитали як пікселі
right = (bins + 0.5) / 1000 * np.array([w, h, w, h])                    # бін → піксель: (b + 0.5) / 1000 · розмір
gt_bus = sparse_gt.boxes[sparse_gt.cats == 9]
vis = P.draw_boxes(im, gt_bus, color=(150, 150, 160), thick=3)
vis = P.draw_boxes(vis, wrong, color=(240, 70, 70))
vis = P.draw_boxes(vis, right, color=(40, 220, 170))
print(f"recall за GT bus: біни як пікселі {P.match_pr(wrong, gt_bus)['tp']}/{len(gt_bus)}, "
      f"після перерахунку {P.match_pr(right, gt_bus)['tp']}/{len(gt_bus)}")
P.show_row([vis], ["сірі — GT bus; червоні — біни 0…999 прочитали як пікселі; бірюзові — перераховані в пікселі кадру"], size=11,
           save=OUT / "02_vlm_coordinate_trap.png")

In [ ]:
# Натовп: скільки машин називає кожен підхід на щільному кадрі. GDINO — для порівняння в тій самій робочій точці.
im = P.imread_rgb(crowd_p)
gt_car = crowd_gt.boxes[np.isin(crowd_gt.cats, [4, 5])]
rows = []
t0 = time.perf_counter()
d = gd(im, "car .", box_thr=0.35)
rows.append({"модель": "Grounding DINO-B", "с на кадр": time.perf_counter() - t0, "боксів": len(d), "є score": "так",
             **P.match_pr(d.boxes, gt_car, crowd_gt.ignore)})
t0 = time.perf_counter()
raw = P.florence_generate(fl_proc, fl, im, "<CAPTION_TO_PHRASE_GROUNDING>", "car", device=DEVICE)
b = (P.parse_loc_tokens(raw)[1] + 0.5) / 1000 * np.array([im.shape[1], im.shape[0]] * 2)
rows.append({"модель": "Florence-2-L", "с на кадр": time.perf_counter() - t0, "боксів": len(b), "є score": "ні",
             **P.match_pr(b, gt_car, crowd_gt.ignore)})
del fl
P.free_memory()

if "qwen2.5-vl" in VLM:
    q_proc, qw = P.load_hf("qwen2.5-vl", DEVICE, dtype=torch.bfloat16 if DEVICE == "cuda" else None)
    prompt = "Locate every car in the image and output the bbox coordinates in JSON format."
    t0 = time.perf_counter()
    raw_q, bq, (wq, hq) = P.qwen_ground(q_proc, qw, im, prompt, device=DEVICE)
    t_q = time.perf_counter() - t0
    bq_px = bq * np.array([im.shape[1] / wq, im.shape[0] / hq] * 2)          # пікселі входу Qwen → пікселі кадру
    print(f"Qwen2.5-VL: вхід моделі {wq}×{hq} замість {im.shape[1]}×{im.shape[0]}. Вихід:\n  {raw_q[:300]}")
    vis = P.draw_boxes(im, gt_car, color=(150, 150, 160), thick=2)
    vis = P.draw_boxes(P.draw_boxes(vis, bq, color=(240, 70, 70)), bq_px, color=(40, 220, 170))
    P.show_row([vis], [f"сірі — GT car; червоні — координати Qwen як є (кадр {wq}×{hq}); бірюзові — перераховані в {im.shape[1]}×{im.shape[0]}"],
               size=12, save=OUT / "02_qwen_resize_trap.png")
    rows.append({"модель": "Qwen2.5-VL-3B", "с на кадр": t_q, "боксів": len(bq), "є score": "ні",
                 **P.match_pr(bq_px, gt_car, crowd_gt.ignore)})
    del qw
    P.free_memory()
vlm = pd.DataFrame(rows).set_index("модель")
vlm["recall"] = vlm.tp / len(gt_car)
print(f"GT машин на кадрі: {len(gt_car)}")
vlm[["с на кадр", "боксів", "tp", "fp", "recall", "є score"]]

**Підсумок розділу.**
* Формат координат — частина моделі: біни Florence-2, прочитані як пікселі, дають нуль влучань; координати Qwen2.5-VL
  живуть у кадрі після його власного ресайзу (з типовим `max_pixels` — 1316 × 728 замість 1920 × 1080).
* Поведінка непередбачувана від моделі до моделі: Florence-2 на щільному кадрі знайшов навіть більше машин, ніж GDINO
  у робочій точці, а Qwen2.5-VL-3B віддав одну рамку на пів кадру. І в жодної немає score — поріг не покрутиш, кривої
  precision / recall не побудуєш.
* Тому VLM — «друга думка» (слайд 42): гнучкий запит із відношеннями й запереченнями для верифікації сумнівних кадрів
  і майнінгу складних випадків на землі, а основні бокси дає OV-детектор.

**Питання в аудиторію:** у якої моделі тайли дали найбільший приріст AP_small і чому — поясніть через розділ 4
(скільки пікселів від цілі лишається на вході кожної моделі)?